In [1]:
import torch
import torch.nn as nn

device = torch.device("mps")
print(f"using device: {device}")

using device: mps


In [2]:
# Toy dimensions

L = 64 # sequence length
N_seq = 8 # number of seq in MSA
c_m = 64 # MSA representation dims
c_z = 32 # pair representation dims

print (f"MSA repr shape: ({N_seq}, {L}, {c_m})")
print (f"Pair repr shape: ({L}, {L}, {c_z})")

MSA repr shape: (8, 64, 64)
Pair repr shape: (64, 64, 32)


In [4]:
aatype = torch.randint(0, 21, (L,), device=device)
residue_index = torch.arange(L, device=device)
msa = torch.randint(0, 23, (N_seq, L), device=device)

print("aatype:        ", aatype.shape)       # (L,)
print("residue_index: ", residue_index.shape) # (L,)
print("msa:           ", msa.shape)           # (N_seq, L)


aatype:         torch.Size([64])
residue_index:  torch.Size([64])
msa:            torch.Size([8, 64])


In [6]:
print(residue_index)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63], device='mps:0')


In [7]:
import torch.nn.functional as F

msa_one_hot = F.one_hot(msa, num_classes=23).float() # (N_seq, L, 23)
print("msa_one_hot:", msa_one_hot.shape)


msa_one_hot: torch.Size([8, 64, 23])


In [9]:
msa

tensor([[ 8,  4, 14,  1, 19, 19, 10,  2,  9,  3,  5,  3,  6, 17,  5,  9, 12,  3,
         19, 16, 17,  1, 10, 19,  1,  7, 20,  0,  9, 12, 19,  7, 20,  0, 13, 10,
         21, 22,  1, 15,  6,  8, 10, 18,  8,  9,  0, 21, 20,  7,  0, 20,  0, 17,
          3, 20,  0,  8, 21,  7,  7,  4, 20, 12],
        [ 4,  9,  3, 19, 20, 13, 21, 12, 18, 15,  2, 19,  5, 10, 14, 13, 13,  9,
         11, 15,  4, 18,  6, 15,  9, 21, 17,  4,  2, 21,  0,  5,  0, 18,  6,  9,
          5, 19, 20,  5,  2, 15, 16, 16,  7, 14, 17, 17, 19, 16,  9,  8, 20,  6,
         12, 14,  0, 22, 19, 15, 14,  9, 21, 12],
        [16,  7, 18, 11, 20, 17, 12, 11,  9, 19, 12, 20,  9,  4, 14,  1, 14, 15,
          0,  0, 17, 14, 21,  8, 18, 18, 20, 17, 22,  8, 15,  1,  8, 19,  3,  0,
          1, 10,  1,  0, 16, 10, 17,  5, 10, 22, 14, 18, 19, 18,  9,  6, 17,  0,
         20, 22,  9,  3, 22, 14, 19,  3,  0,  7],
        [16, 22, 22, 17, 11,  9, 17,  5, 13,  5,  1,  9, 19,  2,  5,  1, 18,  8,
          5,  1, 19,  5, 22, 15, 20, 18,

In [12]:
class InputEmbedder(nn.Module):
    def __init__(self, c_m, c_z):
        super().__init__()
        self.msa_embedder = nn.Linear(23, c_m)
        self.pair_embedder = nn.Linear(65, c_z)

    def forward(self, msa, residue_index):
        # msa: (N_seq, L), residue_index: (L,)

        #MSA
        msa_one_hot = F.one_hot(msa, num_classes=23).float()
        m = self.msa_embedder(msa_one_hot)

        #Pair
        rel_pos = residue_index[None, :] - residue_index[:,None] #(L, L)
        rel_pos_one_hot = F.one_hot(rel_pos.clamp(-32, 32) + 32, num_classes=65).float()
        z = self.pair_embedder(rel_pos_one_hot)

        return m, z
    

embedder = InputEmbedder(c_m, c_z).to(device)
m, z = embedder(msa, residue_index)
print("m:", m.shape)
print("z:", z.shape)

m: torch.Size([8, 64, 64])
z: torch.Size([64, 64, 32])


In [15]:
c_head = 8 # dimensions per attention head
num_heads = 4

to_q = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)
to_k = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)
to_v = nn.Linear(c_m, c_head * num_heads, bias=False).to(device)

pair_bias = nn.Linear(c_z, num_heads, bias=False).to(device)

row = m[0]
q = to_q(row)
k = to_k(row)
v = to_v(row)

print("q:", q.shape)
print("Pair bias input z:", z.shape)
print("Pair bias output:", pair_bias(z).shape)

q: torch.Size([64, 32])
Pair bias input z: torch.Size([64, 64, 32])
Pair bias output: torch.Size([64, 64, 4])


In [16]:
pair_bias

Linear(in_features=32, out_features=4, bias=False)

In [19]:
from einops import rearrange

q = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)
k = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)
v = rearrange(to_q(row), 'l (h d) -> h l d', h = num_heads)

print('q:', q.shape)

# Raw attention scores: for each head, dot every query with every key
scale = c_head ** - 0.5
scores = torch.einsum('h i d, h j d -> h i j', q, k) * scale
print("scores:", scores.shape)

#pair bias
bias = rearrange(pair_bias(z), 'i j h -> h i j')
print("bias:", bias.shape)

scores = scores + bias

q: torch.Size([4, 64, 8])
scores: torch.Size([4, 64, 64])
bias: torch.Size([4, 64, 64])


In [21]:
# softmax over the j axis, for each query pos i, weights over all key positions, j to sum to 1
attn_weights = torch.softmax(scores, dim=-1)
print("attn_weights:", attn_weights.shape)
print("row sums (should all be 1.0):", attn_weights[0, 0].sum().item())


# aggregate values: for each position i, weighted sum of all value vectors
out = torch.einsum('h i j, h j d -> h i d', attn_weights, v)
print("out:", out.shape) # (L, num_heads*c_head)

out = rearrange(out, 'h l d -> l (h d)')
print("out merged:", out.shape)

# final linear projection back to c_m
to_out = nn.Linear(num_heads * c_head, c_m).to(device)
out = to_out(out)
print("out projected:", out.shape)

attn_weights: torch.Size([4, 64, 64])
row sums (should all be 1.0): 1.0000001192092896
out: torch.Size([4, 64, 8])
out merged: torch.Size([64, 32])
out projected: torch.Size([64, 64])


In [23]:
to_gate = nn.Linear(c_m, num_heads * c_head).to(device)

gate = torch.sigmoid(to_gate(row))
gate = rearrange(gate, 'l (h d) -> h l d', h = num_heads)

# recompute out before merging
out_heads = torch.einsum('h i j, h j d -> h i d', attn_weights, v)

# apply gate element wise
out_gated = gate * out_heads

# merge and project
out_gated = rearrange(out_gated, 'h l d -> l (h d)')
out_gated = to_out(out_gated)
print('gated output:', out_gated.shape)

gated output: torch.Size([64, 64])


In [25]:
class MSARowAttention(nn.Module):
    def __init__(self, c_m, c_z, c_head, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_m)
        self.to_q = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_k = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_v = nn.Linear(c_m, c_head * num_heads, bias=False)

        self.to_gate = nn.Linear(c_m, c_head * num_heads)
        self.pair_bias = nn.Linear(c_z, num_heads, bias=False)
        self.to_out = nn.Linear(c_head * num_heads, c_m)

    def forward(self, m, z):
        # m: (N_seq, L, c_m), z: (L, L, c_z)
        m = self.norm(m)
        scale = self.c_head ** -0.5
        bias = rearrange(self.pair_bias(z), 'i j h -> h i j') # (H, L, L)

        q = rearrange(self.to_q(m), 'n l (h d) -> h n l d', h = self.num_heads)
        k = rearrange(self.to_k(m), 'n l (h d) -> h n l d', h = self.num_heads)
        v = rearrange(self.to_v(m), 'n l (h d) -> h n l d', h = self.num_heads)

        scores = torch.einsum('h n i d, h n j d -> h n i j', q, k) * scale
        scores = scores + bias.unsqueeze(1) # broadcast over N_seq
        attn = torch.softmax(scores, dim=-1) # (H, N_seq, L, L)

        out = torch.einsum('h n i j, h n j d -> h n i d', attn, v)
        gate = torch.sigmoid(rearrange(self.to_gate(m), 'n l (h d) -> h n l d', h = self.num_heads))
        out = rearrange(gate * out, 'h n l d -> n l (h d)')
        return self.to_out(out) # (N_seq, L, c_m)

row_attn = MSARowAttention(c_m, c_z, c_head=8, num_heads=4).to(device)
out = row_attn(m,z)
print('output:', out.shape)

output: torch.Size([8, 64, 64])


In [27]:
m_new = m + row_attn(m,z)
print("m_new:", m_new.shape)

m_new: torch.Size([8, 64, 64])


In [ ]:

class MSAColumnAttention(nn.Module):
    def __init__(self, c_m, c_head, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.c_head = c_head
        self.norm = nn.LayerNorm(c_m)
        self.to_q = nn.Linear(c_m, c_head * num_heads, bias=False)
        self.to_k = nn.Linear(c_m, c_head * num_heads, bias=False)
        